# 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1.1 Load Functions

In [ ]:
!pip install category_encoders catboost

In [ ]:
import pickle
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.metrics import precision_recall_curve

from catboost import CatBoostClassifier, Pool
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder

In [ ]:
!python --version

In [ ]:
import sys
import pandas as pd
import numpy as np
import sklearn
import catboost
import category_encoders
import shap
import joblib
import requests

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("catboost:", catboost.__version__)
print("category_encoders:", category_encoders.__version__)
print("shap:", shap.__version__)
print("joblib:", joblib.__version__)
print("requests:", requests.__version__)
import matplotlib
import seaborn as sns

print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)

In [ ]:
requirements = """pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
catboost==1.2.10
category-encoders==2.8.1
shap==0.51.0
joblib==1.5.3
requests==2.32.4
matplotlib==3.10.0
seaborn==0.13.2
"""

with open("/content/drive/MyDrive/Thesis/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt saved to Google Drive")

## 1.2 Load Split Data and Preprocessing Objects

In [ ]:
# Load classification split data
split_path = "/content/drive/MyDrive/Thesis/model_splits_preprocessed.pkl"

with open(split_path, "rb") as f:
    split_data = pickle.load(f)

print("Loaded keys:")
print(split_data.keys())

In [ ]:
# Feature lists
features_no_ses = split_data["features_no_ses"]
features_with_ses = split_data["features_with_ses"]

categorical_features = split_data["categorical_features"]
numeric_features_ses = split_data["numeric_features_ses"]

high_cardinality_features_no_ses = split_data["high_cardinality_features_no_ses"]
high_cardinality_features_ses = split_data["high_cardinality_features_ses"]

low_cardinality_features_no_ses = split_data["low_cardinality_features_no_ses"]
low_cardinality_features_ses = split_data["low_cardinality_features_ses"]

# Standard sklearn modeling data
X_train_no_ses = split_data["X_train_no_ses"]
X_test_no_ses = split_data["X_test_no_ses"]
y_train_no_ses = split_data["y_train_no_ses"]
y_test_no_ses = split_data["y_test_no_ses"]

X_train_ses = split_data["X_train_ses"]
X_test_ses = split_data["X_test_ses"]
y_train_ses = split_data["y_train_ses"]
y_test_ses = split_data["y_test_ses"]

# CatBoost modeling data
X_train_cb_clf_no_ses = split_data["X_train_cb_clf_no_ses"]
X_test_cb_clf_no_ses = split_data["X_test_cb_clf_no_ses"]
cat_cols_clf_no_ses = split_data["cat_cols_clf_no_ses"]
cat_features_clf_no_ses = split_data["cat_features_clf_no_ses"]

X_train_cb_clf_ses = split_data["X_train_cb_clf_ses"]
X_test_cb_clf_ses = split_data["X_test_cb_clf_ses"]
cat_cols_clf_ses = split_data["cat_cols_clf_ses"]
cat_features_clf_ses = split_data["cat_features_clf_ses"]

In [ ]:
print("No SES")
print("X_train_no_ses:", X_train_no_ses.shape)
print("X_test_no_ses:", X_test_no_ses.shape)
print("y_train_no_ses:", y_train_no_ses.shape)
print("y_test_no_ses:", y_test_no_ses.shape)

print("\nWith SES")
print("X_train_ses:", X_train_ses.shape)
print("X_test_ses:", X_test_ses.shape)
print("y_train_ses:", y_train_ses.shape)
print("y_test_ses:", y_test_ses.shape)

print("\nCatBoost No SES")
print("X_train_cb_clf_no_ses:", X_train_cb_clf_no_ses.shape)
print("X_test_cb_clf_no_ses:", X_test_cb_clf_no_ses.shape)
print("cat_features_clf_no_ses:", cat_features_clf_no_ses)

print("\nCatBoost With SES")
print("X_train_cb_clf_ses:", X_train_cb_clf_ses.shape)
print("X_test_cb_clf_ses:", X_test_cb_clf_ses.shape)
print("cat_features_clf_ses:", cat_features_clf_ses)

In [ ]:
print("Missing SES numeric values in CatBoost train:")
print(X_train_cb_clf_ses[numeric_features_ses].isna().sum())

print("\nMissing SES numeric values in CatBoost test:")
print(X_test_cb_clf_ses[numeric_features_ses].isna().sum())

In [ ]:


numeric_transformer_scaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

numeric_transformer_unscaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

logreg_preprocessor_no_ses = ColumnTransformer(
    transformers=[
        ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_no_ses),
        ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_no_ses)
    ],
    remainder="drop"
)

logreg_preprocessor_ses = ColumnTransformer(
    transformers=[
        ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_ses),
        ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_ses),
        ("num", numeric_transformer_scaled, numeric_features_ses)
    ],
    remainder="drop"
)

rf_clf_preprocessor_no_ses = ColumnTransformer(
    transformers=[
        ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_no_ses),
        ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_no_ses)
    ],
    remainder="drop"
)

rf_clf_preprocessor_ses = ColumnTransformer(
    transformers=[
        ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_ses),
        ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_ses),
        ("num", numeric_transformer_unscaled, numeric_features_ses)
    ],
    remainder="drop"
)

# 2. Evaluation Functions

In [ ]:
def evaluate_classification_model(model, X_test, y_test, model_name, feature_set):
    y_pred = model.predict(X_test)
    y_pred = np.array(y_pred).ravel()

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = None

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results = {
        "stage": "Classification",
        "feature_set": feature_set,
        "model": model_name,

        # Main classification metrics
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),

        # Confusion matrix counts
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,

        # Error-pattern metrics
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
    }

    if y_proba is not None:
        results["roc_auc"] = roc_auc_score(y_test, y_proba)
        results["pr_auc"] = average_precision_score(y_test, y_proba)
    else:
        results["roc_auc"] = np.nan
        results["pr_auc"] = np.nan

    print(f"\n===== {model_name} | {feature_set} =====")
    print(pd.Series(results))

    print("\nConfusion Matrix:")
    print(pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Actual 0 Short-stay", "Actual 1 Long-stay"],
        columns=["Predicted 0 Short-stay", "Predicted 1 Long-stay"]
    ))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return results

# 3. Modeling


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## 3.0 Majority Baseline

In [ ]:
save_dir = "/content/drive/MyDrive/Thesis"

majority_baseline = DummyClassifier(strategy="most_frequent")
majority_baseline.fit(X_train_ses, y_train_ses)

y_pred_majority = majority_baseline.predict(X_test_ses)
y_proba_majority = majority_baseline.predict_proba(X_test_ses)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test_ses, y_pred_majority).ravel()

majority_baseline_results = pd.DataFrame([{
    "stage": "Classification",
    "feature_set": "No features",
    "model": "Majority Baseline",
    "accuracy": accuracy_score(y_test_ses, y_pred_majority),
    "precision": precision_score(y_test_ses, y_pred_majority, zero_division=0),
    "recall": recall_score(y_test_ses, y_pred_majority, zero_division=0),
    "f1": f1_score(y_test_ses, y_pred_majority, zero_division=0),
    "roc_auc": roc_auc_score(y_test_ses, y_proba_majority),
    "pr_auc": average_precision_score(y_test_ses, y_proba_majority),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
    "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else np.nan
}])

majority_baseline_results.to_csv(
    os.path.join(save_dir, "majority_baseline_test_metrics.csv"),
    index=False
)

majority_baseline_results.round(3)

## 3.1 Logistic Regression

- baseline
- No hyperparameter tuning

In [ ]:
# Logistic Regression - No SES
logreg_clf_no_ses = Pipeline(
    steps=[
        ("preprocessor", logreg_preprocessor_no_ses),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

# Logistic Regression - With SES
logreg_clf_ses = Pipeline(
    steps=[
        ("preprocessor", logreg_preprocessor_ses),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

In [ ]:
# Fit Logistic Regression models
logreg_clf_no_ses.fit(X_train_no_ses, y_train_no_ses)
logreg_clf_ses.fit(X_train_ses, y_train_ses)

In [ ]:
classification_results = []

classification_results.append(
    evaluate_classification_model(
        logreg_clf_no_ses,
        X_test_no_ses,
        y_test_no_ses,
        "Logistic Regression",
        "No SES"
    )
)

classification_results.append(
    evaluate_classification_model(
        logreg_clf_ses,
        X_test_ses,
        y_test_ses,
        "Logistic Regression",
        "With SES"
    )
)

In [ ]:
# ============================================================
# Save Logistic Regression outputs
# Compatible with your current pipeline:
# ("preprocessor", ...)
# ("classifier", LogisticRegression(...))
# ============================================================

save_dir = "/content/drive/MyDrive/Thesis"
os.makedirs(save_dir, exist_ok=True)


# ============================================================
# Helper: get feature names from your Logistic Regression pipeline
# ============================================================

def get_logreg_feature_names(pipeline):
    preprocessor = pipeline.named_steps["preprocessor"]

    feature_names = []

    for name, transformer, columns in preprocessor.transformers_:
        if transformer == "drop":
            continue

        # TargetEncoder: output keeps original column name
        if name == "target_breed":
            feature_names.extend(list(columns))

        # OneHotEncoder
        elif name == "onehot_cat":
            onehot_features = transformer.get_feature_names_out(columns)
            feature_names.extend(list(onehot_features))

        # Numeric pipeline
        elif name == "num":
            feature_names.extend(list(columns))

        else:
            try:
                names = transformer.get_feature_names_out(columns)
                feature_names.extend(list(names))
            except Exception:
                feature_names.extend(list(columns))

    return feature_names


# ============================================================
# Helper: save one Logistic Regression model
# ============================================================

def save_logreg_outputs(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    model_name,
    feature_set,
    filename_prefix,
    save_dir
):
    # -------------------------
    # Predictions
    # -------------------------
    y_pred = np.array(model.predict(X_test)).ravel()
    y_proba = model.predict_proba(X_test)[:, 1]

    # -------------------------
    # Metrics
    # -------------------------
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    test_metrics = {
        "stage": "Classification",
        "feature_set": feature_set,
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else np.nan
    }

    # -------------------------
    # Save model
    # -------------------------
    joblib.dump(
        model,
        f"{save_dir}/{filename_prefix}.pkl"
    )

    # -------------------------
    # Save test metrics
    # -------------------------
    pd.DataFrame([test_metrics]).to_csv(
        f"{save_dir}/{filename_prefix}_test_metrics.csv",
        index=False
    )

    # -------------------------
    # Save predictions
    # -------------------------
    pred_df = pd.DataFrame({
        "index": X_test.index,
        "y_true": np.array(y_test),
        "y_pred": y_pred,
        "y_proba": y_proba
    })

    pred_df.to_csv(
        f"{save_dir}/{filename_prefix}_test_predictions.csv",
        index=False
    )

    # -------------------------
    # Save confusion matrix
    # -------------------------
    cm_df = pd.DataFrame(
        cm,
        index=["Actual short-stay", "Actual long-stay"],
        columns=["Predicted short-stay", "Predicted long-stay"]
    )

    cm_df.to_csv(
        f"{save_dir}/{filename_prefix}_confusion_matrix.csv"
    )

    # -------------------------
    # Save classification report
    # -------------------------
    report_df = pd.DataFrame(
        classification_report(
            y_test,
            y_pred,
            output_dict=True,
            zero_division=0
        )
    ).transpose()

    report_df.to_csv(
        f"{save_dir}/{filename_prefix}_classification_report.csv"
    )

    # -------------------------
    # Save coefficients
    # -------------------------
    feature_names = get_logreg_feature_names(model)
    coefficients = model.named_steps["classifier"].coef_.ravel()

    if len(feature_names) != len(coefficients):
        print("WARNING: feature name / coefficient length mismatch")
        print("n feature_names:", len(feature_names))
        print("n coefficients:", len(coefficients))
        feature_names = [f"feature_{i}" for i in range(len(coefficients))]

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
        "abs_coefficient": np.abs(coefficients)
    }).sort_values("abs_coefficient", ascending=False)

    coef_df.to_csv(
        f"{save_dir}/{filename_prefix}_coefficients.csv",
        index=False
    )

    # -------------------------
    # Save summary
    # -------------------------
    summary = {
        "model": model_name,
        "feature_set": feature_set,
        "test_metrics": test_metrics,
        "saved_outputs": {
            "model": f"{filename_prefix}.pkl",
            "test_metrics": f"{filename_prefix}_test_metrics.csv",
            "test_predictions": f"{filename_prefix}_test_predictions.csv",
            "confusion_matrix": f"{filename_prefix}_confusion_matrix.csv",
            "classification_report": f"{filename_prefix}_classification_report.csv",
            "coefficients": f"{filename_prefix}_coefficients.csv"
        }
    }

    with open(f"{save_dir}/{filename_prefix}_summary.json", "w") as f:
        json.dump(summary, f, indent=4, default=str)

    # -------------------------
    # Save run metadata
    # -------------------------
    classifier = model.named_steps["classifier"]

    run_metadata = {
        "model": "LogisticRegression",
        "feature_set": feature_set,
        "n_train": int(X_train.shape[0]),
        "n_test": int(X_test.shape[0]),
        "n_features_raw": int(X_train.shape[1]),
        "raw_features": list(X_train.columns),
        "n_features_after_preprocessing": int(len(feature_names)),
        "pipeline_steps": list(model.named_steps.keys()),
        "class_weight": classifier.get_params().get("class_weight"),
        "max_iter": classifier.get_params().get("max_iter"),
        "solver": classifier.get_params().get("solver"),
        "penalty": classifier.get_params().get("penalty"),
        "C": classifier.get_params().get("C"),
        "random_state": classifier.get_params().get("random_state")
    }

    with open(f"{save_dir}/{filename_prefix}_run_metadata.json", "w") as f:
        json.dump(run_metadata, f, indent=4, default=str)

    print(f"Saved Logistic Regression outputs: {feature_set}")
    print(f"Prefix: {filename_prefix}")
    display(pd.DataFrame([test_metrics]))
    display(coef_df.head(20))

    return test_metrics, pred_df, cm_df, report_df, coef_df

In [ ]:
# ============================================================
# Save Logistic Regression - No SES
# ============================================================

logreg_no_ses_outputs = save_logreg_outputs(
    model=logreg_clf_no_ses,
    X_train=X_train_no_ses,
    X_test=X_test_no_ses,
    y_train=y_train_no_ses,
    y_test=y_test_no_ses,
    model_name="Logistic Regression",
    feature_set="No SES",
    filename_prefix="logreg_classifier_no_ses",
    save_dir=save_dir
)


# ============================================================
# Save Logistic Regression - With SES
# ============================================================

logreg_ses_outputs = save_logreg_outputs(
    model=logreg_clf_ses,
    X_train=X_train_ses,
    X_test=X_test_ses,
    y_train=y_train_ses,
    y_test=y_test_ses,
    model_name="Logistic Regression",
    feature_set="With SES",
    filename_prefix="logreg_classifier_ses",
    save_dir=save_dir
)

In [ ]:
for f in sorted(os.listdir(save_dir)):
    if "logreg_classifier" in f:
        print(f)

## 3.2 Random Forest Classifier

- Secondary: ROC-AUC, PR-AUC, Precision

In [ ]:
rf_clf_no_ses = Pipeline(
    steps=[
        ("preprocess", rf_clf_preprocessor_no_ses),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_clf_ses = Pipeline(
    steps=[
        ("preprocess", rf_clf_preprocessor_ses),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [ ]:
# Random Forest Search Space

rf_param_dist = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [15, 25, 35],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": ["balanced", "balanced_subsample"]
}

### 3.2.1 RF with SES

In [ ]:
# With SES

# 1. Fit tuned RF with SES
rf_search_ses = RandomizedSearchCV(
    estimator=rf_clf_ses,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="f1",
    cv=cv,
    random_state=42,
    verbose=2,
    n_jobs=1,
    return_train_score=True
)

rf_search_ses.fit(X_train_ses, y_train_ses)

print("Best CV F1:", rf_search_ses.best_score_)
print("Best parameters:")
print(rf_search_ses.best_params_)

# 2. Save best model
joblib.dump(
    rf_search_ses.best_estimator_,
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses.pkl"
)

# 3. Save full CV results
pd.DataFrame(rf_search_ses.cv_results_).to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuning_cv_results_ses.csv",
    index=False
)

# 4. Save best params and best CV score
rf_ses_summary = {
    "model": "Random Forest Classifier",
    "feature_set": "With SES",
    "best_cv_f1": rf_search_ses.best_score_,
    "best_params": rf_search_ses.best_params_
}

with open("/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_summary.json", "w") as f:
    json.dump(rf_ses_summary, f, indent=4)

# 5. Evaluate on held-out test set
rf_ses_test_result = evaluate_classification_model(
    rf_search_ses.best_estimator_,
    X_test_ses,
    y_test_ses,
    "Random Forest Tuned",
    "With SES"
)

pd.DataFrame([rf_ses_test_result]).to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_test_metrics.csv",
    index=False
)

# 6. Save test predictions and predicted probabilities
y_pred_rf_ses = rf_search_ses.best_estimator_.predict(X_test_ses)
y_proba_rf_ses = rf_search_ses.best_estimator_.predict_proba(X_test_ses)[:, 1]

rf_ses_predictions = pd.DataFrame({
    "index": X_test_ses.index,
    "y_true": np.array(y_test_ses),
    "y_pred": np.array(y_pred_rf_ses).ravel(),
    "y_proba": y_proba_rf_ses
})

rf_ses_predictions.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_test_predictions.csv",
    index=False
)

# 7. Save confusion matrix
cm_rf_ses = confusion_matrix(y_test_ses, y_pred_rf_ses)

cm_rf_ses_df = pd.DataFrame(
    cm_rf_ses,
    index=["Actual short-stay", "Actual long-stay"],
    columns=["Predicted short-stay", "Predicted long-stay"]
)

cm_rf_ses_df.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_confusion_matrix.csv"
)

# 8. Save feature importance for tuned RF with SES

best_rf_ses = rf_search_ses.best_estimator_

preprocessor = best_rf_ses.named_steps["preprocess"]
rf_model = best_rf_ses.named_steps["model"]

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance_df.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_feature_importance.csv",
    index=False
)

feature_importance_df.head(20)

In [ ]:
# 2. Save best model
joblib.dump(
    rf_search_ses.best_estimator_,
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses.pkl"
)

# 3. Save full CV results
pd.DataFrame(rf_search_ses.cv_results_).to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuning_cv_results_ses.csv",
    index=False
)

# 4. Save best params and best CV score
rf_ses_summary = {
    "model": "Random Forest Classifier",
    "feature_set": "With SES",
    "best_cv_f1": rf_search_ses.best_score_,
    "best_params": rf_search_ses.best_params_
}

with open("/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_summary.json", "w") as f:
    json.dump(rf_ses_summary, f, indent=4)

# 5. Evaluate on held-out test set
rf_ses_test_result = evaluate_classification_model(
    rf_search_ses.best_estimator_,
    X_test_ses,
    y_test_ses,
    "Random Forest Tuned",
    "With SES"
)

pd.DataFrame([rf_ses_test_result]).to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_test_metrics.csv",
    index=False
)

# 6. Save test predictions and predicted probabilities
y_pred_rf_ses = rf_search_ses.best_estimator_.predict(X_test_ses)
y_proba_rf_ses = rf_search_ses.best_estimator_.predict_proba(X_test_ses)[:, 1]

rf_ses_predictions = pd.DataFrame({
    "index": X_test_ses.index,
    "y_true": np.array(y_test_ses),
    "y_pred": np.array(y_pred_rf_ses).ravel(),
    "y_proba": y_proba_rf_ses
})

rf_ses_predictions.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_test_predictions.csv",
    index=False
)

# 7. Save confusion matrix
cm_rf_ses = confusion_matrix(y_test_ses, y_pred_rf_ses)

cm_rf_ses_df = pd.DataFrame(
    cm_rf_ses,
    index=["Actual short-stay", "Actual long-stay"],
    columns=["Predicted short-stay", "Predicted long-stay"]
)

cm_rf_ses_df.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_confusion_matrix.csv"
)

# 8. Save feature importance for tuned RF with SES

best_rf_ses = rf_search_ses.best_estimator_

preprocessor = best_rf_ses.named_steps["preprocess"]
rf_model = best_rf_ses.named_steps["model"]

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance_df.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_ses_feature_importance.csv",
    index=False
)

feature_importance_df.head(20)

### 3.2.2 RF Without SES

In [ ]:
# =========================
# Random Forest Tuning - No SES
# =========================

rf_search_no_ses = RandomizedSearchCV(
    estimator=rf_clf_no_ses,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="f1",
    cv=cv,
    random_state=42,
    verbose=2,
    n_jobs=1,
    return_train_score=True
)

rf_search_no_ses.fit(X_train_no_ses, y_train_no_ses)

print("Best CV F1:", rf_search_no_ses.best_score_)
print("Best parameters:")
print(rf_search_no_ses.best_params_)

# -------------------------
# Save best model
# -------------------------

joblib.dump(
    rf_search_no_ses.best_estimator_,
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_no_ses.pkl"
)

# -------------------------
# Save CV tuning results
# -------------------------

pd.DataFrame(rf_search_no_ses.cv_results_).to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuning_cv_results_no_ses.csv",
    index=False
)

# -------------------------
# Save tuning summary
# -------------------------

rf_no_ses_summary = {
    "model": "Random Forest Classifier",
    "feature_set": "No SES",
    "best_cv_f1": rf_search_no_ses.best_score_,
    "best_params": rf_search_no_ses.best_params_
}

with open("/content/drive/MyDrive/Thesis/rf_classifier_tuned_no_ses_summary.json", "w") as f:
    json.dump(rf_no_ses_summary, f, indent=4)

# -------------------------
# Evaluate on held-out test set
# -------------------------

rf_no_ses_test_result = evaluate_classification_model(
    rf_search_no_ses.best_estimator_,
    X_test_no_ses,
    y_test_no_ses,
    "Random Forest Tuned",
    "No SES"
)

pd.DataFrame([rf_no_ses_test_result]).to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_no_ses_test_metrics.csv",
    index=False
)

# -------------------------
# Save test predictions
# -------------------------

y_pred_rf_no_ses = rf_search_no_ses.best_estimator_.predict(X_test_no_ses)
y_proba_rf_no_ses = rf_search_no_ses.best_estimator_.predict_proba(X_test_no_ses)[:, 1]

rf_no_ses_predictions = pd.DataFrame({
    "index": X_test_no_ses.index,
    "y_true": np.array(y_test_no_ses),
    "y_pred": np.array(y_pred_rf_no_ses).ravel(),
    "y_proba": y_proba_rf_no_ses
})

rf_no_ses_predictions.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_no_ses_test_predictions.csv",
    index=False
)

# -------------------------
# Save confusion matrix
# -------------------------

cm_rf_no_ses = confusion_matrix(y_test_no_ses, y_pred_rf_no_ses)

cm_rf_no_ses_df = pd.DataFrame(
    cm_rf_no_ses,
    index=["Actual short-stay", "Actual long-stay"],
    columns=["Predicted short-stay", "Predicted long-stay"]
)

cm_rf_no_ses_df.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_no_ses_confusion_matrix.csv"
)

# -------------------------
# Save feature importance
# -------------------------

best_rf_no_ses = rf_search_no_ses.best_estimator_

preprocessor = best_rf_no_ses.named_steps["preprocess"]
rf_model = best_rf_no_ses.named_steps["model"]

try:
    feature_names = preprocessor.get_feature_names_out()
except:
    target_features = high_cardinality_features_no_ses

    onehot_encoder = preprocessor.named_transformers_["onehot_cat"]
    onehot_features = onehot_encoder.get_feature_names_out(low_cardinality_features_no_ses)

    feature_names = list(target_features) + list(onehot_features)

feature_importance_df_no_ses = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance_df_no_ses.to_csv(
    "/content/drive/MyDrive/Thesis/rf_classifier_tuned_no_ses_feature_importance.csv",
    index=False
)

print("Saved all Random Forest no-SES outputs.")
display(feature_importance_df_no_ses.head(20))

## 3.3 CatBoost Classifier

Use categorical columns

### 3.3.0 CatBoost Setup

In [ ]:
# Use categorical feature indices saved from preprocessing
cat_features_cls_no_ses = cat_features_clf_no_ses
cat_features_cls_ses = cat_features_clf_ses

print("No-SES categorical columns:")
print(X_train_cb_clf_no_ses.iloc[:, cat_features_cls_no_ses].columns.tolist())

print("\nWith-SES categorical columns:")
print(X_train_cb_clf_ses.iloc[:, cat_features_cls_ses].columns.tolist())

In [ ]:
print("Missing values in CatBoost no-SES train:")
print(X_train_cb_clf_no_ses.isna().sum().sort_values(ascending=False).head(10))

print("\nMissing values in CatBoost with-SES train:")
print(X_train_cb_clf_ses.isna().sum().sort_values(ascending=False).head(10))

print("\nMissing values in CatBoost with-SES test:")
print(X_test_cb_clf_ses.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
print(X_train_cb_clf_ses[numeric_features_ses].isna().sum())
print(X_test_cb_clf_ses[numeric_features_ses].isna().sum())

In [ ]:
for col in cat_cols_clf_no_ses:
    X_train_cb_clf_no_ses[col] = X_train_cb_clf_no_ses[col].fillna("Unknown").astype(str)
    X_test_cb_clf_no_ses[col] = X_test_cb_clf_no_ses[col].fillna("Unknown").astype(str)

for col in cat_cols_clf_ses:
    X_train_cb_clf_ses[col] = X_train_cb_clf_ses[col].fillna("Unknown").astype(str)
    X_test_cb_clf_ses[col] = X_test_cb_clf_ses[col].fillna("Unknown").astype(str)

In [ ]:
# With SES
train_pool_ses = Pool(
    data=X_train_cb_clf_ses,
    label=y_train_ses,
    cat_features=cat_features_clf_ses
)

test_pool_ses = Pool(
    data=X_test_cb_clf_ses,
    label=y_test_ses,
    cat_features=cat_features_clf_ses
)

# No SES
train_pool_no_ses = Pool(
    data=X_train_cb_clf_no_ses,
    label=y_train_no_ses,
    cat_features=cat_features_clf_no_ses
)

test_pool_no_ses = Pool(
    data=X_test_cb_clf_no_ses,
    label=y_test_no_ses,
    cat_features=cat_features_clf_no_ses
)

In [ ]:
print("CatBoost no-SES categorical indices:", cat_features_clf_no_ses)
print("CatBoost with-SES categorical indices:", cat_features_clf_ses)

print("No SES train shape:", X_train_cb_clf_no_ses.shape, y_train_no_ses.shape)
print("No SES test shape:", X_test_cb_clf_no_ses.shape, y_test_no_ses.shape)

print("With SES train shape:", X_train_cb_clf_ses.shape, y_train_ses.shape)
print("With SES test shape:", X_test_cb_clf_ses.shape, y_test_ses.shape)

print("\nWith SES numeric missing:")
print(X_train_cb_clf_ses[numeric_features_ses].isna().sum())
print(X_test_cb_clf_ses[numeric_features_ses].isna().sum())

In [ ]:
# CatBoost base model - No SES
catboost_clf_no_ses = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="F1",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
    task_type="GPU",
    devices="0"
)

# CatBoost base model - With SES
catboost_clf_ses = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="F1",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
    task_type="GPU",
    devices="0"
)

# CatBoost hyperparameter search space
cat_param_dist = {
    "iterations": [300, 500, 800],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [1, 3, 5, 7, 10],
    "border_count": [32, 64, 128]
}

### 3.3.1 Cat With SES

In [ ]:
# Save directory
save_dir = "/content/drive/MyDrive/Thesis"
os.makedirs(save_dir, exist_ok=True)

# =========================
# CatBoost Tuning - With SES
# =========================

cat_search_result_ses = catboost_clf_ses.randomized_search(
    param_distributions=cat_param_dist,
    X=train_pool_ses,
    n_iter=30,
    cv=5,
    partition_random_seed=42,
    stratified=True,
    shuffle=True,
    refit=True,
    verbose=True
)

best_params_ses = cat_search_result_ses["params"]

print("Best parameters - CatBoost With SES:")
print(best_params_ses)

# -------------------------
# Save tuned CatBoost model
# -------------------------

catboost_clf_ses.save_model(
    f"{save_dir}/catboost_classifier_tuned_ses.cbm"
)

joblib.dump(
    catboost_clf_ses,
    f"{save_dir}/catboost_classifier_tuned_ses.pkl"
)

# -------------------------
# Save search result object
# -------------------------

with open(f"{save_dir}/catboost_classifier_search_result_ses.pkl", "wb") as f:
    pickle.dump(cat_search_result_ses, f)

# -------------------------
# Save CV results if available
# -------------------------

if isinstance(cat_search_result_ses, dict) and "cv_results" in cat_search_result_ses:
    pd.DataFrame(cat_search_result_ses["cv_results"]).to_csv(
        f"{save_dir}/catboost_classifier_tuning_cv_results_ses.csv",
        index=False
    )

# -------------------------
# Evaluate on held-out test set
# -------------------------

cat_ses_test_result = evaluate_classification_model(
    catboost_clf_ses,
    test_pool_ses,
    y_test_ses,
    "CatBoost Tuned",
    "With SES"
)

pd.DataFrame([cat_ses_test_result]).to_csv(
    f"{save_dir}/catboost_classifier_tuned_ses_test_metrics.csv",
    index=False
)

# -------------------------
# Save test predictions
# -------------------------

y_pred_cat_ses = catboost_clf_ses.predict(test_pool_ses)
y_pred_cat_ses = np.array(y_pred_cat_ses).ravel()

y_proba_cat_ses = catboost_clf_ses.predict_proba(test_pool_ses)[:, 1]

cat_ses_predictions = pd.DataFrame({
    "index": X_test_cb_clf_ses.index,
    "y_true": np.array(y_test_ses),
    "y_pred": y_pred_cat_ses,
    "y_proba": y_proba_cat_ses
})

cat_ses_predictions.to_csv(
    f"{save_dir}/catboost_classifier_tuned_ses_test_predictions.csv",
    index=False
)

# -------------------------
# Save confusion matrix
# -------------------------

cm_cat_ses = confusion_matrix(y_test_ses, y_pred_cat_ses)

cm_cat_ses_df = pd.DataFrame(
    cm_cat_ses,
    index=["Actual short-stay", "Actual long-stay"],
    columns=["Predicted short-stay", "Predicted long-stay"]
)

cm_cat_ses_df.to_csv(
    f"{save_dir}/catboost_classifier_tuned_ses_confusion_matrix.csv"
)

# -------------------------
# Save feature importance
# -------------------------

cat_ses_feature_importance = pd.DataFrame({
    "feature": X_train_cb_clf_ses.columns,
    "importance": catboost_clf_ses.get_feature_importance(train_pool_ses)
}).sort_values("importance", ascending=False)

cat_ses_feature_importance.to_csv(
    f"{save_dir}/catboost_classifier_tuned_ses_feature_importance.csv",
    index=False
)

# -------------------------
# Save summary
# -------------------------

cat_ses_summary = {
    "model": "CatBoost Classifier",
    "feature_set": "With SES",
    "best_params": best_params_ses,
    "test_metrics": cat_ses_test_result,
    "search_result_keys": list(cat_search_result_ses.keys()) if isinstance(cat_search_result_ses, dict) else None
}

with open(f"{save_dir}/catboost_classifier_tuned_ses_summary.json", "w") as f:
    json.dump(cat_ses_summary, f, indent=4)

print("Saved all CatBoost with-SES outputs.")
display(cat_ses_feature_importance.head(20))

# -------------------------
# Save report
# -------------------------

from sklearn.metrics import classification_report

classification_report_dict = classification_report(
    y_test_ses,
    y_pred_cat_ses,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(classification_report_dict).transpose().to_csv(
    f"{save_dir}/catboost_classifier_tuned_ses_classification_report.csv"
)

pd.DataFrame([best_params_ses]).to_csv(
    f"{save_dir}/catboost_classifier_tuned_ses_best_params.csv",
    index=False
)

run_metadata = {
    "model": "CatBoostClassifier",
    "feature_set": "With SES",
    "n_train": int(X_train_cb_clf_ses.shape[0]),
    "n_test": int(X_test_cb_clf_ses.shape[0]),
    "n_features": int(X_train_cb_clf_ses.shape[1]),
    "features": list(X_train_cb_clf_ses.columns),
    "cat_features": cat_features_clf_ses,
    "n_iter": 30,
    "cv": 5,
    "random_seed": 42,
    "task_type": "GPU"
}

with open(f"{save_dir}/catboost_classifier_tuned_ses_run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=4, default=str)

In [ ]:
# -------------------------
# Save summary
# -------------------------

cat_ses_summary = {
    "model": "CatBoost Classifier",
    "feature_set": "With SES",
    "best_params": best_params_ses,
    "test_metrics": cat_ses_test_result,
    "search_result_keys": list(cat_search_result_ses.keys()) if isinstance(cat_search_result_ses, dict) else None
}

with open(f"{save_dir}/catboost_classifier_tuned_ses_summary.json", "w") as f:
    json.dump(cat_ses_summary, f, indent=4, default=str)

print("Saved all CatBoost with-SES outputs.")
display(cat_ses_feature_importance.head(20))

# -------------------------
# Save report
# -------------------------

from sklearn.metrics import classification_report

classification_report_dict = classification_report(
    y_test_ses,
    y_pred_cat_ses,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(classification_report_dict).transpose().to_csv(
    f"{save_dir}/catboost_classifier_tuned_ses_classification_report.csv"
)

pd.DataFrame([best_params_ses]).to_csv(
    f"{save_dir}/catboost_classifier_tuned_ses_best_params.csv",
    index=False
)

run_metadata = {
    "model": "CatBoostClassifier",
    "feature_set": "With SES",
    "n_train": int(X_train_cb_clf_ses.shape[0]),
    "n_test": int(X_test_cb_clf_ses.shape[0]),
    "n_features": int(X_train_cb_clf_ses.shape[1]),
    "features": list(X_train_cb_clf_ses.columns),
    "cat_features": cat_features_clf_ses,
    "n_iter": 30,
    "cv": 5,
    "random_seed": 42,
    "task_type": "GPU"
}

with open(f"{save_dir}/catboost_classifier_tuned_ses_run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=4, default=str)


### 3.3.2 Without SES

In [ ]:
# Save directory
save_dir = "/content/drive/MyDrive/Thesis"
os.makedirs(save_dir, exist_ok=True)

# =========================
# CatBoost Tuning - No SES
# =========================

cat_search_result_no_ses = catboost_clf_no_ses.randomized_search(
    param_distributions=cat_param_dist,
    X=train_pool_no_ses,
    n_iter=30,
    cv=5,
    partition_random_seed=42,
    stratified=True,
    shuffle=True,
    refit=True,
    verbose=True
)

best_params_no_ses = cat_search_result_no_ses["params"]

print("Best parameters - CatBoost No SES:")
print(best_params_no_ses)

# -------------------------
# Save tuned CatBoost model
# -------------------------

catboost_clf_no_ses.save_model(
    f"{save_dir}/catboost_classifier_tuned_no_ses.cbm"
)

joblib.dump(
    catboost_clf_no_ses,
    f"{save_dir}/catboost_classifier_tuned_no_ses.pkl"
)

# -------------------------
# Save search result object
# -------------------------

with open(f"{save_dir}/catboost_classifier_search_result_no_ses.pkl", "wb") as f:
    pickle.dump(cat_search_result_no_ses, f)

# -------------------------
# Save CV results if available
# -------------------------

if isinstance(cat_search_result_no_ses, dict) and "cv_results" in cat_search_result_no_ses:
    pd.DataFrame(cat_search_result_no_ses["cv_results"]).to_csv(
        f"{save_dir}/catboost_classifier_tuning_cv_results_no_ses.csv",
        index=False
    )

# -------------------------
# Evaluate on held-out test set
# -------------------------

cat_no_ses_test_result = evaluate_classification_model(
    catboost_clf_no_ses,
    test_pool_no_ses,
    y_test_no_ses,
    "CatBoost Tuned",
    "No SES"
)

pd.DataFrame([cat_no_ses_test_result]).to_csv(
    f"{save_dir}/catboost_classifier_tuned_no_ses_test_metrics.csv",
    index=False
)

# -------------------------
# Save test predictions
# -------------------------

y_pred_cat_no_ses = catboost_clf_no_ses.predict(test_pool_no_ses)
y_pred_cat_no_ses = np.array(y_pred_cat_no_ses).ravel()

y_proba_cat_no_ses = catboost_clf_no_ses.predict_proba(test_pool_no_ses)[:, 1]

cat_no_ses_predictions = pd.DataFrame({
    "index": X_test_cb_clf_no_ses.index,
    "y_true": np.array(y_test_no_ses),
    "y_pred": y_pred_cat_no_ses,
    "y_proba": y_proba_cat_no_ses
})

cat_no_ses_predictions.to_csv(
    f"{save_dir}/catboost_classifier_tuned_no_ses_test_predictions.csv",
    index=False
)

# -------------------------
# Save confusion matrix
# -------------------------

cm_cat_no_ses = confusion_matrix(y_test_no_ses, y_pred_cat_no_ses)

cm_cat_no_ses_df = pd.DataFrame(
    cm_cat_no_ses,
    index=["Actual short-stay", "Actual long-stay"],
    columns=["Predicted short-stay", "Predicted long-stay"]
)

cm_cat_no_ses_df.to_csv(
    f"{save_dir}/catboost_classifier_tuned_no_ses_confusion_matrix.csv"
)

# -------------------------
# Save feature importance
# -------------------------

cat_no_ses_feature_importance = pd.DataFrame({
    "feature": X_train_cb_clf_no_ses.columns,
    "importance": catboost_clf_no_ses.get_feature_importance(train_pool_no_ses)
}).sort_values("importance", ascending=False)

cat_no_ses_feature_importance.to_csv(
    f"{save_dir}/catboost_classifier_tuned_no_ses_feature_importance.csv",
    index=False
)

# -------------------------
# Save summary
# -------------------------

cat_no_ses_summary = {
    "model": "CatBoost Classifier",
    "feature_set": "No SES",
    "best_params": best_params_no_ses,
    "test_metrics": cat_no_ses_test_result,
    "search_result_keys": list(cat_search_result_no_ses.keys()) if isinstance(cat_search_result_no_ses, dict) else None
}

with open(f"{save_dir}/catboost_classifier_tuned_no_ses_summary.json", "w") as f:
    json.dump(cat_no_ses_summary, f, indent=4, default=str)

print("Saved all CatBoost no-SES outputs.")
display(cat_no_ses_feature_importance.head(20))

# -------------------------
# Save report
# -------------------------

from sklearn.metrics import classification_report

classification_report_dict = classification_report(
    y_test_no_ses,
    y_pred_cat_no_ses,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(classification_report_dict).transpose().to_csv(
    f"{save_dir}/catboost_classifier_tuned_no_ses_classification_report.csv"
)

pd.DataFrame([best_params_no_ses]).to_csv(
    f"{save_dir}/catboost_classifier_tuned_no_ses_best_params.csv",
    index=False
)

run_metadata = {
    "model": "CatBoostClassifier",
    "feature_set": "No SES",
    "n_train": int(X_train_cb_clf_no_ses.shape[0]),
    "n_test": int(X_test_cb_clf_no_ses.shape[0]),
    "n_features": int(X_train_cb_clf_no_ses.shape[1]),
    "features": list(X_train_cb_clf_no_ses.columns),
    "cat_features": cat_features_clf_no_ses,
    "n_iter": 30,
    "cv": 5,
    "random_seed": 42,
    "task_type": "GPU"
}

with open(f"{save_dir}/catboost_classifier_tuned_no_ses_run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=4, default=str)

# 4. Results

In [ ]:
save_dir = "/content/drive/MyDrive/Thesis"

metric_files = {
    "CatBoost No SES": f"{save_dir}/catboost_classifier_tuned_no_ses_test_metrics.csv",
    "CatBoost With SES": f"{save_dir}/catboost_classifier_tuned_ses_test_metrics.csv",
    "Random Forest No SES": f"{save_dir}/rf_classifier_tuned_no_ses_test_metrics.csv",
    "Random Forest With SES": f"{save_dir}/rf_classifier_tuned_ses_test_metrics.csv",
    "Logistic Regression No SES": f"{save_dir}/logreg_classifier_no_ses_test_metrics.csv",
    "Logistic Regression With SES": f"{save_dir}/logreg_classifier_ses_test_metrics.csv",
}

results_list = []

for name, path in metric_files.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        df["result_file"] = name
        results_list.append(df)
    else:
        print("Missing file:", path)

classification_final_results = pd.concat(results_list, ignore_index=True)

cols_order = [
    "stage", "feature_set", "model",
    "accuracy", "precision", "recall", "f1",
    "roc_auc", "pr_auc",
    "tn", "fp", "fn", "tp",
    "false_positive_rate", "false_negative_rate"
]

existing_cols = [col for col in cols_order if col in classification_final_results.columns]

classification_final_results = classification_final_results[existing_cols].sort_values(
    by="f1",
    ascending=False
)

classification_final_results